# A DQN that learns from rewards — Rainbow-lite on CartPole

_Capstones_

---

This notebook is a practice scaffold for the **A DQN that learns from rewards — Rainbow-lite on CartPole** lesson. We build the agent one improvement at a time — the base DQN loop, the Double-DQN target, the Dueling head, and Prioritized replay — then race the combined agent against a vanilla baseline. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — gymnasium + PyTorch (Colab)

### Step 1 — Import the tools and sanity-check the dueling formula

We pin every random seed so the run is reproducible, and fix the discount factor `GAMMA`. Before building anything, we verify the **dueling** aggregation by hand: a dueling head splits the Q-value into a state-value `V(s)` and per-action advantages `A(s,a)`, then recombines them as `Q = V + (A - mean_a A)`. Subtracting the mean advantage removes the redundancy between V and A so the two streams stay identifiable.

In [ ]:
# In Colab first run:  !pip install gymnasium
# (torch is preinstalled in Colab; gymnasium is not.)
import random
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym

torch.manual_seed(0)
random.seed(0)
np.random.seed(0)
GAMMA = 0.99

# --- Sanity-check the DUELING head's Eq. 9 aggregation: Q = V + (A - mean_a A). ---
V = torch.tensor([[10.0]])                 # state-value V(s), shape [1,1]
A = torch.tensor([[6.0, 0.0]])             # advantage A(s,a) for two actions, shape [1,2]
mean_A = A.mean(dim=1, keepdim=True)        # mean advantage across actions = 3
Q = V + (A - mean_A)                        # Q = 10 + (6-3, 0-3) = [13, 7]
print("dueling worked example:  V =", V.item(), " A =", A.tolist()[0],
      " mean(A) =", A.mean().item(), " Q = V+(A-mean) =", Q.tolist()[0])
# dueling worked example:  V = 10.0  A = [6.0, 0.0]  mean(A) = 3.0  Q = V+(A-mean) = [13.0, 7.0]

### Step 2 — Build the network and the replay buffer

`QNet` is the Q-network (COMPONENT 3, Dueling DQN). A shared torso feeds either a single Q head (plain DQN) or a value stream plus an advantage stream that recombine with the Eq. 9 formula we just checked. `PrioritizedReplay` (COMPONENT 4) stores past transitions; when `per_on=True` it samples transitions in proportion to their TD-error priority and returns importance-sampling weights to correct the resulting bias. With `per_on=False` it degrades to the uniform buffer of the original DQN.

In [ ]:
# ============================================================================
# COMPONENT 3 (Dueling DQN): a Q-network whose head optionally splits into a
# value stream V(s) and an advantage stream A(s,a), recombined by Eq. 9.
# ============================================================================
class QNet(nn.Module):
    def __init__(self, obs_dim, n_act, hidden=128, dueling=True):
        super().__init__()
        self.dueling = dueling
        self.torso = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(),
                                   nn.Linear(hidden, hidden), nn.ReLU())
        if dueling:
            self.value_head = nn.Linear(hidden, 1)        # V(s):    one scalar per state
            self.adv_head   = nn.Linear(hidden, n_act)    # A(s,a):  one number per action
        else:
            self.q_head = nn.Linear(hidden, n_act)        # plain DQN: Q(s,a) directly
    def forward(self, x):
        h = self.torso(x)
        if self.dueling:
            v = self.value_head(h)                        # [batch, 1]
            a = self.adv_head(h)                          # [batch, n_act]
            return v + (a - a.mean(dim=1, keepdim=True))  # Eq. 9 dueling aggregation
        return self.q_head(h)


# ============================================================================
# COMPONENT 4 (Prioritized Experience Replay): sample by TD-error priority,
# return importance-sampling (IS) weights. With per_on=False it is a plain
# uniform replay buffer (every transition equally likely, weights = 1).
# ============================================================================
class PrioritizedReplay:
    def __init__(self, capacity=50000, alpha=0.6, per_on=True):
        self.capacity = capacity
        self.alpha = alpha
        self.per_on = per_on
        self.buf = []
        self.prios = np.zeros((capacity,), dtype=np.float32)
        self.pos = 0
    def push(self, s, a, r, s2, done):
        max_p = self.prios.max() if self.buf else 1.0    # new transitions get max priority
        if len(self.buf) < self.capacity:
            self.buf.append((s, a, r, s2, done))
        else:
            self.buf[self.pos] = (s, a, r, s2, done)
        self.prios[self.pos] = max_p
        self.pos = (self.pos + 1) % self.capacity
    def sample(self, n, beta=0.4):
        N = len(self.buf)
        if self.per_on:
            prios = self.prios[:N] ** self.alpha          # p_i^alpha
            P = prios / prios.sum()                       # P(i) = p_i^a / sum_k p_k^a  (Eq. 1)
            idx = np.random.choice(N, n, p=P)
            w = (N * P[idx]) ** (-beta)                   # w_i = (1/(N*P(i)))^beta  (S 3.4)
            w = w / w.max()                               # normalize for stability
        else:
            idx = np.random.choice(N, n)                  # uniform: the original DQN buffer
            w = np.ones(n, dtype=np.float32)
        s, a, r, s2, d = zip(*[self.buf[i] for i in idx])
        return (torch.tensor(np.array(s), dtype=torch.float32),
                torch.tensor(a, dtype=torch.long),
                torch.tensor(r, dtype=torch.float32),
                torch.tensor(np.array(s2), dtype=torch.float32),
                torch.tensor(d, dtype=torch.float32),
                idx, torch.tensor(w, dtype=torch.float32))
    def update_priorities(self, idx, td_errors):
        for i, e in zip(idx, td_errors):
            self.prios[i] = abs(float(e)) + 1e-5          # p_i = |TD error| + epsilon
    def __len__(self):
        return len(self.buf)

### Step 3 — Define one learning update

`learn` performs a single gradient step (COMPONENT 1 base loop + COMPONENT 2 Double-DQN target). It samples a batch, computes the online network's Q for the actions taken, and builds the bootstrap target. With **Double DQN** the online network *selects* the next action while the target network *evaluates* it — decoupling selection from evaluation curbs the overestimation bias of plain DQN. The squared TD error is weighted by the importance-sampling weights, and the priorities are refreshed from the fresh TD errors.

In [ ]:
# ============================================================================
# COMPONENT 1 (base DQN loop) + COMPONENT 2 (Double DQN target).
# One update: build the target, weight the squared TD loss by IS weights,
# step, and refresh priorities.
# ============================================================================
def learn(q, q_target, opt, replay, batch=64, beta=0.4, use_double=True):
    if len(replay) < batch:
        return
    s, a, r, s2, done, idx, w = replay.sample(batch, beta=beta)
    q_sa = q(s).gather(1, a.unsqueeze(1)).squeeze(1)        # Q_online(s,a) for the action taken
    with torch.no_grad():
        if use_double:                                     # DOUBLE: online selects, target evaluates
            next_a = q(s2).argmax(1, keepdim=True)         # argmax_a Q_online(s2, a)
            q_next = q_target(s2).gather(1, next_a).squeeze(1)
        else:                                              # plain DQN: target net selects AND evaluates
            q_next = q_target(s2).max(1).values
        y = r + GAMMA * (1.0 - done) * q_next              # TD target (1-done zeros terminal bootstrap)
    td = q_sa - y                                          # TD error
    loss = (w * td.pow(2)).mean()                          # IS-weighted squared TD loss
    opt.zero_grad()
    loss.backward()
    opt.step()
    replay.update_priorities(idx, td.detach().cpu().numpy())

### Step 4 — Assemble the agent and race it against the baseline

`train` wires the pieces together on CartPole: an online net, a target net that starts as a copy and is periodically re-synced, epsilon-greedy exploration that decays over time, and the IS exponent `beta` annealed from 0.4 toward 1.0. The three toggles switch the improvements on or off. The final run trains the combined **Rainbow-lite** agent and a **vanilla DQN** (all toggles off) under identical settings so any difference is attributable to the improvements, not luck.

In [ ]:
# ============================================================================
# Train the assembled agent on CartPole. Toggles select which improvements
# are active. The base target network is always on (it IS the DQN baseline).
# ============================================================================
def train(use_double=True, use_dueling=True, use_per=True,
          episodes=400, sync_every=200, label=""):
    torch.manual_seed(0)
    random.seed(0)
    np.random.seed(0)
    env = gym.make("CartPole-v1")
    obs_dim = env.observation_space.shape[0]
    n_act = env.action_space.n
    q        = QNet(obs_dim, n_act, dueling=use_dueling)
    q_target = QNet(obs_dim, n_act, dueling=use_dueling)
    q_target.load_state_dict(q.state_dict())               # target net starts as a copy
    opt = torch.optim.Adam(q.parameters(), lr=1e-3)
    replay = PrioritizedReplay(per_on=use_per)
    eps, step, recent, history = 1.0, 0, [], []
    print(f"[{label}] double={use_double} dueling={use_dueling} per={use_per}")
    for ep in range(episodes):
        beta = min(1.0, 0.4 + 0.6 * ep / episodes)         # anneal IS exponent beta 0.4 -> 1.0
        s, _ = env.reset(seed=ep)
        done = False
        ep_ret = 0.0
        while not done:
            if random.random() < eps:                      # epsilon-greedy exploration
                a = env.action_space.sample()
            else:
                with torch.no_grad():
                    a = int(q(torch.tensor(s, dtype=torch.float32).unsqueeze(0)).argmax())
            s2, rew, term, trunc, _ = env.step(a)
            done = term or trunc
            replay.push(s, a, rew, s2, float(done))
            s = s2
            ep_ret += rew
            step += 1
            learn(q, q_target, opt, replay, beta=beta, use_double=use_double)
            if step % sync_every == 0:                     # periodic target-net sync
                q_target.load_state_dict(q.state_dict())
        eps = max(0.02, eps * 0.97)                         # decay exploration
        recent.append(ep_ret)
        avg = sum(recent[-100:]) / len(recent[-100:])
        history.append(avg)
        if ep % 20 == 0:
            print(f"  ep {ep:3d}  eps {eps:.2f}  avg return (last 100): {avg:6.1f}")
        if len(recent) >= 100 and avg >= 475:
            print(f"  -> SOLVED CartPole at episode {ep} (avg return {avg:.1f} >= 475).")
            break
    env.close()
    return history


# --- FINAL RUN: combined "Rainbow-lite" agent vs the vanilla DQN baseline. ---
print("\n=== Combined agent (Double + Dueling + Prioritized replay) ===")
combined_hist = train(use_double=True, use_dueling=True, use_per=True, label="Rainbow-lite")

print("\n=== Vanilla DQN baseline (all three toggles OFF) ===")
vanilla_hist  = train(use_double=False, use_dueling=False, use_per=False, label="Vanilla DQN")

print("\nCombined avg-return trajectory:", [round(h, 1) for h in combined_hist[::20]])
print("Vanilla  avg-return trajectory:", [round(h, 1) for h in vanilla_hist[::20]])
# The combined agent climbs to ~500 and SOLVES CartPole noticeably faster and more stably
# than the vanilla DQN. Flip any single toggle to isolate that improvement's effect.
# (Exact numbers vary by hardware/seed; our small run, NOT the paper's Atari results.)

## Visualize the data & results

_Does combining all four improvements — DQN base + Double-DQN target + Dueling head + Prioritized replay — make the agent solve CartPole faster and more stably than a vanilla DQN? We train our combined 'Rainbow-lite' agent and a vanilla DQN (all toggles off) with identical net size, optimizer, learning rate, and seed, and plot the average episode return._

### Step 5 — Read the comparison curves

The two histories above are the average return (over the last 100 episodes) per episode for each agent. The combined agent should cross the 475 "solved" line first and then hold near 500, while the vanilla DQN learns the same task more slowly and more noisily. The sketch below restates how the curves are produced; flip one toggle at a time to attribute the gain to a single improvement.

In [ ]:
# Sketch of how the two curves above are produced (full loop in the CODE cell).
# Train the combined agent and the vanilla DQN on CartPole-v1 for the same number of
# episodes with identical net size / optimizer / lr / epsilon schedule / seed,
# recording the average return (last 100 episodes) per episode.
#
#   combined_hist = train(use_double=True,  use_dueling=True,  use_per=True)   # green: solves ~ep 140
#   vanilla_hist  = train(use_double=False, use_dueling=False, use_per=False)  # blue:  slower climb
#
# Combined (Double + Dueling + PER) -> crosses the 475 "solved" line first, then holds near 500.
# Vanilla DQN -> learns the same task but takes longer and is noisier.
# Toggle ONE improvement at a time (e.g. use_double=True only) to attribute the gain.
# (Numbers are from our own run; the papers report Atari scores, not these CartPole values.)